# Xori Hori — Llama 3.2 3B + LoRA для Cloudflare

Эта версия обучает LoRA именно для `meta-llama/Llama-3.2-3B-Instruct`, чтобы адаптер соответствовал Cloudflare Workers AI. Нужен официальный доступ к gated-модели Meta через Hugging Face; обход доступа не предусмотрен.


In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.51,<5" "peft>=0.16,<0.19" "accelerate>=1.2" datasets safetensors huggingface_hub
print('Установка завершена. Теперь один раз: Runtime → Restart session. Затем запускай следующую ячейку.')


In [ ]:
import os, json, shutil, subprocess
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

BASE='meta-llama/Llama-3.2-3B-Instruct'
REPO_URL='https://github.com/xoristalin-dotcom/Xori-.git'
WORK_ROOT=Path('/content')
WORK=WORK_ROOT/'Xori-'
OUTPUT_DIR=WORK_ROOT/'xori_llama32_3b_lora_cloudflare'
MAX_LEN=768
SYSTEM=('Ты — Хори Кёко из Horimiya. Отвечай по-русски естественно, прямо и по-человечески. '
        'Не копируй реплики из произведения. Не выдумывай мысли, действия или факты о собеседнике. '
        'Не форсируй дружбу или романтику. Собеседник сам управляет своими действиями и словами. '
        'Сохраняй характер Хори: ответственная, заботливая, прямая, иногда вспыльчивая, упрямая и с чувством юмора. '
        'Обычно отвечай 2–5 предложениями и не задавай больше одного вопроса.')

if not torch.cuda.is_available(): raise RuntimeError('GPU не найден. Выбери GPU в Colab.')
print('PyTorch:',torch.__version__); print('GPU:',torch.cuda.get_device_name(0)); print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))

# Проверяем официальный HF-доступ ДО скачивания нескольких GB модели.
hf_token=os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    if not hf_token:
        hf_token=userdata.get('HF_TOKEN')
except Exception:
    pass
if not hf_token:
    raise RuntimeError('Не найден HF_TOKEN. Добавь свой Hugging Face User Access Token в Colab Secrets как HF_TOKEN и перезапусти session.')

from huggingface_hub import HfApi
api=HfApi(token=hf_token)
try:
    api.model_info(BASE)
except Exception as e:
    raise RuntimeError('Нет подтвержденного доступа к gated-модели Llama 3.2 3B. Сначала получи официальный доступ Meta/Hugging Face. Исходная ошибка: '+str(e))
print('HF access: OK')

if WORK.exists(): shutil.rmtree(WORK,ignore_errors=True)
p=subprocess.run(['git','clone','--depth','1',REPO_URL,str(WORK)],capture_output=True,text=True)
if p.returncode!=0: raise RuntimeError('Git clone failed: '+p.stderr)

pairs=[]
seed=WORK/'hori_sft_seed.jsonl'
if seed.exists():
    for line in seed.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row=json.loads(line)
            if row.get('user') and row.get('assistant'): pairs.append((row['user'].strip(),row['assistant'].strip()))
training=WORK/'hori_training.json'
if training.exists():
    data=json.loads(training.read_text(encoding='utf-8'))
    for row in data.get('examples',[]):
        if row.get('approved') and row.get('user') and row.get('assistant'): pairs.append((row['user'].strip(),row['assistant'].strip()))
pairs=list(dict.fromkeys(pairs))
print('Training pairs:',len(pairs))
if not pairs: raise RuntimeError('Датасет пуст.')

tokenizer=AutoTokenizer.from_pretrained(BASE,token=hf_token)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token

def encode_pair(u,a):
    messages=[{'role':'system','content':SYSTEM},{'role':'user','content':u}]
    prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    pids=tokenizer(prompt,add_special_tokens=False)['input_ids']
    aids=tokenizer(a,add_special_tokens=False)['input_ids']
    eos=[tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    ids=pids+aids+eos; labels=[-100]*len(pids)+aids+eos
    if len(ids)>MAX_LEN: ids=ids[:MAX_LEN]; labels=labels[:MAX_LEN]
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}
dataset=Dataset.from_list([encode_pair(u,a) for u,a in pairs])
print(dataset)

use_bf16=torch.cuda.is_bf16_supported(); dtype=torch.bfloat16 if use_bf16 else torch.float16
print('dtype:',dtype)
model=AutoModelForCausalLM.from_pretrained(BASE,token=hf_token,dtype=dtype,low_cpu_mem_usage=True)
model.config.use_cache=False
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

# r=8 — совместимо с ограничениями Cloudflare LoRA.
lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj'])
model=get_peft_model(model,lora)
model.print_trainable_parameters()

OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
args=TrainingArguments(output_dir=str(OUTPUT_DIR),num_train_epochs=4,per_device_train_batch_size=1,gradient_accumulation_steps=4,learning_rate=1e-4,warmup_ratio=0.05,weight_decay=0.01,logging_steps=1,save_strategy='epoch',save_total_limit=1,report_to='none',fp16=not use_bf16,bf16=use_bf16,gradient_checkpointing=True,optim='adamw_torch',remove_unused_columns=False)
collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,padding=True,label_pad_token_id=-100,return_tensors='pt')
trainer=Trainer(model=model,args=args,train_dataset=dataset,data_collator=collator)
print('=== START TRAINING ===')
trainer.train()

trainer.save_model(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))
adapter=OUTPUT_DIR/'adapter_model.safetensors'; config=OUTPUT_DIR/'adapter_config.json'
if not adapter.exists() or not config.exists(): raise RuntimeError('LoRA-файлы не созданы.')

# Cloudflare требует model_type=llama в adapter_config.json.
cfg=json.loads(config.read_text(encoding='utf-8'))
cfg['model_type']='llama'
config.write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')

zip_path=shutil.make_archive(str(WORK_ROOT/'xori_llama32_3b_lora_cloudflare'),'zip',root_dir=str(OUTPUT_DIR))
print('=== ГОТОВО ==='); print('ZIP:',zip_path); print('Adapter MB:',round(adapter.stat().st_size/1024/1024,2)); print('Cloudflare model_type:',cfg.get('model_type'))


После обучения ZIP появится в `/content` как `xori_llama32_3b_lora_cloudflare.zip`.

Важно: 25 примеров — только технический прогон. Для финальной качества Xori нужно значительно расширить датасет.

После успешного обучения adapter можно готовить к загрузке в Cloudflare Workers AI на совместимую Llama 3.2 3B базу.